In [19]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecVideoRecorder
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import wandb
from wandb.integration.sb3 import WandbCallback
import os
import glob

In [20]:
total_timesteps = 200_000

In [ ]:
class VideoLoggerCallback(BaseCallback):
    def __init__(self, video_folder, verbose=0):
        super().__init__(verbose)
        self.video_folder = video_folder
        self.logged_videos = set()

    def _on_step(self) -> bool:
        for video_path in glob.glob(os.path.join(self.video_folder, "*.mp4")):
            if video_path not in self.logged_videos and os.path.getsize(video_path) > 0:
                wandb.log({"training_video": wandb.Video(video_path, fps=30, format="mp4")})
                self.logged_videos.add(video_path)
        return True

In [23]:
# Initialize a new wandb run


config = {
    "total_timesteps": total_timesteps,
    "env_name": "CartPole-v1",
    "algorithm": "PPO",
    "policy_type": "MlpPolicy",
}



run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="RL_locomotion_FEKTBOT",
    # Set the wandb project where this run will be logged.
    project="RL_locomotion_FEKTBOT_project",
    # Track hyperparameters and run metadata.
    config=config,
    sync_tensorboard=True,   
    monitor_gym=True,        
    save_code=True,
)

wandb: WARNING Artifact "source-RL_locomotion_FEKTBOT_project-_home_aizej_Desktop_RL_lokomoce_FEKTBOT_testing_gymnasium.ipynb" already exists with the same content. No new version will be created.


In [24]:
def make_single_env():
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    env = gym.wrappers.RecordEpisodeStatistics(env)
    return env


# Create a vectorized environment
env = SubprocVecEnv([make_single_env for _ in range(8)])  # 8 parallel processes

env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda step: step % 10000 == 0,
    video_length=200,
)

In [25]:



model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=f"runs/{run.id}")



model.learn(total_timesteps=total_timesteps, 
            callback=CallbackList([WandbCallback(
                gradient_save_freq=100,
                model_save_path=f"models/{run.id}",
                verbose=2,
            ), VideoLoggerCallback(video_folder=f"videos/{run.id}")]
            ))



Using cpu device


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Logging to runs/q3w6x3s2/PPO_1
Saving video to /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-0-to-step-200.mp4
MoviePy - Building video /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-0-to-step-200.mp4.
MoviePy - Writing video /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-0-to-step-200.mp4



MoviePy - Done !
MoviePy - video ready /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-0-to-step-200.mp4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.5     |
|    ep_rew_mean     | 21.5     |
| time/              |          |
|    fps             | 2623     |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 37.3        |
|    ep_rew_mean          | 37.3        |
| time/                   |             |
|    fps                  | 3060        |
|    iterations           | 2           |
|    time_elapsed         | 10          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.014992962 |
|    clip_fraction        | 0.264       |
|    clip_range           | 0

MoviePy - Done !
MoviePy - video ready /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-10000-to-step-10200.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 196         |
|    ep_rew_mean          | 196         |
| time/                   |             |
|    fps                  | 2855        |
|    iterations           | 5           |
|    time_elapsed         | 28          |
|    total_timesteps      | 81920       |
| train/                  |             |
|    approx_kl            | 0.007952362 |
|    clip_fraction        | 0.118       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.549      |
|    explained_variance   | 0.562       |
|    learning_rate        | 0.0003      |
|    loss                 | 12.4        |
|    n_updates            | 40          |
|    policy_gradient_loss | -0.0174     |
|    value_loss           | 35.3        |
---------------------------

MoviePy - Done !
MoviePy - video ready /home/aizej/Desktop/RL_lokomoce_FEKTBOT/videos/q3w6x3s2/rl-video-step-20000-to-step-20200.mp4
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 446          |
|    ep_rew_mean          | 446          |
| time/                   |              |
|    fps                  | 2759         |
|    iterations           | 10           |
|    time_elapsed         | 59           |
|    total_timesteps      | 163840       |
| train/                  |              |
|    approx_kl            | 0.0075934194 |
|    clip_fraction        | 0.109        |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.457       |
|    explained_variance   | 0.99         |
|    learning_rate        | 0.0003       |
|    loss                 | 0.145        |
|    n_updates            | 90           |
|    policy_gradient_loss | -0.0107      |
|    value_loss           | 0.896        |
-------

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
